# Data Contracts and SQL Warehouse

Build a local relational warehouse, enforce integrity constraints, reconcile SQL and Pandas counts, and record lineage. The SQLite binary is generated locally and ignored by Git; SQL, contracts, portable CSV reports, and this executed notebook are committed.

In [1]:
from pathlib import Path
import subprocess, sys, json, base64
import pandas as pd
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/processed/train.csv").exists()), None)
if ROOT is None:
    checkout = Path("/content/scientsbank_portfolio")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Junghwan0516/science-answer-benchmark.git",
                        str(checkout)], check=True)
    ROOT = checkout
sys.path.insert(0, str(ROOT / "scripts"))
print("Project:", ROOT.name)

Project: science-answer-benchmark


In [2]:
from build_warehouse import build
quality = build(ROOT)
print("Quality checks:", len(quality["checks"]))

PASS: complete coverage, unchanged labels, reproducible assignments, zero question/input overlap, versioned hashes, and training-only split generation.
{
  "contract_version": "1.0",
  "all_passed": true,
  "table_counts": {
    "domains": 12,
    "labels": 5,
    "questions": 135,
    "answers": 4969,
    "answer_features": 4969,
    "development_assignments": 4969
  },
  "checks": [
    "source record/question/domain counts",
    "source primary key and allowed labels",
    "nonempty source text",
    "single context per question",
    "feature primary key and complete one-to-one coverage",
    "finite/range-constrained features",
    "word-count and reference-overlap reconciliation",
    "development assignment preservation and disjointness",
    "SQLite primary/foreign/check constraints",
    "SQLite foreign-key and integrity checks",
    "SQL/Pandas label-count reconciliation",
    "SQL/Pandas question-count reconciliation"
  ],
  "official_test_files_read": false,
  "model_traini

## Relational tables

Domains and labels are dimensions. Questions contain one context/reference. Answers link to questions and labels. Feature rows and development assignments link one-to-one to answers. Primary keys, foreign keys, ranges, text non-emptiness, and split-weight checks are enforced by SQLite.

In [3]:
import sqlite3
connection = sqlite3.connect(ROOT / "data/warehouse/scientsbank.sqlite")
print(pd.read_sql_query("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name", connection).to_string(index=False))

                   name  type
        answer_features table
                answers table
development_assignments table
                domains table
                 labels table
              questions table
       v_domain_summary  view
        v_label_summary  view
    v_partition_summary  view
     v_question_metrics  view


## SQL: domain-level composition

This query joins question, answer, and feature tables through a documented view. It returns observed label shares, not model performance.

In [4]:
query = "SELECT * FROM v_domain_summary ORDER BY answer_count DESC"
print(query)
print(pd.read_sql_query(query, connection).to_string(index=False))

SELECT * FROM v_domain_summary ORDER BY answer_count DESC
domain  answer_count  question_count  correct_count  observed_correct_share  mean_answer_words
    ME           828              23            377                0.455314          11.754831
    MX           697              20            231                0.331420          13.368723
    PS           545              11            249                0.456881          10.860550
    SE           539              15            220                0.408163          16.571429
    EM           430              12            170                0.395349          12.404651
    VB           396              11            164                0.414141          12.449495
    LF           393              11             91                0.231552          14.470738
    FN           323               9            170                0.526316          17.544892
    ST           283               8            107                0.378092           6

## SQL: split coverage and rare labels

In [5]:
print(pd.read_sql_query("SELECT * FROM v_partition_summary ORDER BY internal_split", connection).to_string(index=False))
print("Foreign-key violations:", connection.execute("PRAGMA foreign_key_check").fetchall())
print("Integrity:", connection.execute("PRAGMA integrity_check").fetchone()[0])
connection.close()

internal_split  answer_count  question_count  non_domain_count
   calibration           718              20                 3
   development           715              20                 3
           fit          3536              95                17
Foreign-key violations: []
Integrity: ok


## Lineage and reproducibility

SHA-256 hashes pin the source CSV, derived features, assignments, schema, and query definitions. The builder creates a temporary database, verifies it, and atomically replaces the local warehouse only on success. Unknown labels, orphan records, missing feature rows, and invalid ranges are rejected. These are local batch-pipeline controls; no cloud orchestration, production deployment, or SLA is claimed.

In [6]:
lineage = json.loads((ROOT / "reports/engineering/lineage.json").read_text())
print(json.dumps(lineage, indent=2))

{
  "warehouse": "data/warehouse/scientsbank.sqlite (locally rebuilt; not committed)",
  "inputs_sha256": {
    "data/processed/train.csv": "4654218036087a13d8e15e83d6060a60729f32a734c8f61720513e3e897d42e2",
    "data/processed/training_features.csv": "476156be5b1765bbef4b6a228cd28f7213c6b0e353756d9051a5ab71365a59fd",
    "data/splits/v1/answer_assignments.csv": "ee1312fe017bb559fdda54e7b31846e1b89db3797abfb6ae3a0b7c20a53da5fa",
    "data/contracts/training_contract.json": "a106e3212296cb046d29cb1c5c27050a4fe58b5861bdc89553baae23637e587e",
    "sql/01_schema.sql": "988f42359751fcdcb10672800cadf2ad75c69d28b7abf92881f28f1784f9627d",
    "sql/02_analytics.sql": "f20e6cb48142bdde23a668f69eaa3e46f6950cd6bf5199986d2a32d8c7a4d141"
  },
  "source": "SciEntsBank official training Core only",
  "transformations": [
    "derive documented text features",
    "preserve source text and labels",
    "normalize question/answer/features/assignment tables",
    "enforce relational constraints",
    "ma

## Rebuild command

`python scripts/run_pipeline.py --stage advanced` rebuilds feature/statistics/SQL outputs, executes these notebooks, runs the checks, and verifies release hashes. Use `--refresh-checksums` only after reviewing intentional artifact changes. `--stage full` also rebuilds the original audit and versioned split outputs. No prediction model is trained.

Source data and data-derived outputs: CC BY-SA 3.0; Dzikovska et al. (2013).